In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

1. Create DataFrame

In [0]:
from pyspark.sql.functions import *
df=spark.createDataFrame(data,columns)


2. Add required derived columns

In [0]:
df = df.withColumn("test_cost", col("tests_count")*2000)\
       .withColumn("total_bill",
                   col("consultation_fee")+col("test_cost"))

3. Filter High-Value Patients

In [0]:
high_value = df.filter(col("total_bill") > 8000)
high_value.show()

+--------+------------+---------+-----------+----------------+-----------+---------+----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|test_cost|total_bill|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|     4000|      9000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|     2000|      9000|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|     6000|      9000|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+



4. Aggregation by Department

In [0]:
dept_summary = df.groupBy("department")\
    .agg(sum("total_bill").alias("revenue"))
dept_summary.show()

+-----------+-------+
| department|revenue|
+-----------+-------+
| Cardiology|  23000|
|Orthopedics|  16000|
|Dermatology|   9000|
|  Neurology|   9000|
+-----------+-------+



 5. Sort Results

In [0]:
dept_summary.orderBy(desc("revenue")).show()

+-----------+-------+
| department|revenue|
+-----------+-------+
| Cardiology|  23000|
|Orthopedics|  16000|
|Dermatology|   9000|
|  Neurology|   9000|
+-----------+-------+



 PART 2 — SPARK SQL
 1. Convert DataFrame to a temp view

In [0]:
df.createOrReplaceTempView("patient_visits")

2. Write SQL to:
Fetch specific department records
Calculate revenue per city
Identify top patients
Count patients per department

In [0]:
%sql
SELECT *
FROM patient_visits
WHERE department='Cardiology';

visit_id,patient_name,city,department,consultation_fee,tests_count,test_cost,total_bill
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,2000,7000
104,Priya Nair,Bangalore,Cardiology,5000,2,4000,9000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,2000,7000


In [0]:
%sql
SELECT city,
SUM(total_bill) AS revenue
FROM patient_visits
GROUP BY city;

city,revenue
Hyderabad,7000
Delhi,7000
Mumbai,3500
Bangalore,14500
Chennai,9000
Kolkata,9000
Ahmedabad,7000


In [0]:
%sql
SELECT *
FROM patient_visits
ORDER BY total_bill DESC
LIMIT 5;

visit_id,patient_name,city,department,consultation_fee,tests_count,test_cost,total_bill
106,Ananya Das,Kolkata,Orthopedics,3000,3,6000,9000
105,Vikram Singh,Chennai,Neurology,7000,1,2000,9000
104,Priya Nair,Bangalore,Cardiology,5000,2,4000,9000
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,7000
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,2000,7000


In [0]:
%sql
SELECT department, COUNT(*) AS patients
FROM patient_visits
GROUP BY department;

department,patients
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


PART 3 — DELTA LAKE CORE
1. Create a Delta table from the dataset

In [0]:
df.write.format("delta").saveAsTable("patient_delta")

2. Insert new records

In [0]:
%sql
INSERT INTO patient_delta
VALUES (109,'New Patient','Delhi','Cardiology',4000,1,2000,6000);

num_affected_rows,num_inserted_rows
1,1


3. Update existing records

In [0]:
%sql
UPDATE patient_delta
SET consultation_fee=6000
WHERE visit_id=101;

num_affected_rows
1


4. Delete specific records

In [0]:
%sql
DELETE FROM patient_delta
WHERE city='Ahmedabad';

num_affected_rows
1


5. Perform an UPSERT using MERGE

In [0]:
%sql
MERGE INTO patient_delta t
USING updates s
ON t.visit_id=s.visit_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4795362430170909>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'MERGE INTO patient_delta t\nUSING updates s\nON t.visit_id=s.visit_id\nWHEN MATCHED THEN UPDATE SET *\nWHEN NOT MATCHED THEN INSERT *;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.

PART 4 — DELTA ADVANCED
1. Retrieve table history

In [0]:
%sql
DESCRIBE HISTORY patient_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-05-04T04:19:00.000Z,141477211871293,azuser6403_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2018233997510340),86588a28-4f64-4f1f-bf57-d880b86cf734,0504-040911-6ssd4rmw-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2852, p25FileSize -> 2815, numDeletionVectorsRemoved -> 1, minFileSize -> 2815, numAddedFiles -> 1, maxFileSize -> 2815, p75FileSize -> 2815, p50FileSize -> 2815, numAddedBytes -> 2815)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-04T04:18:58.000Z,141477211871293,azuser6403_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(city#14440 = Ahmedabad)""])",null,List(2018233997510340),86588a28-4f64-4f1f-bf57-d880b86cf734,0504-040911-6ssd4rmw-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1503, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1088, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 413)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-04T04:17:33.000Z,141477211871293,azuser6403_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2018233997510340),12bbc3b4-0e25-463b-b36d-9bb022ec5aa5,0504-040911-6ssd4rmw-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7386, p25FileSize -> 2852, numDeletionVectorsRemoved -> 1, minFileSize -> 2852, numAddedFiles -> 1, maxFileSize -> 2852, p75FileSize -> 2852, p50FileSize -> 2852, numAddedBytes -> 2852)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-04T04:17:31.000Z,141477211871293,azuser6403_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(visit_id#13822L = 101)""])",null,List(2018233997510340),12bbc3b4-0e25-463b-b36d-9bb022ec5aa5,0504-040911-6ssd4rmw-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3970, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2309, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2321, rewriteTimeMs -> 1636)",null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-05-04T04:17:16.000Z,141477211871293,azuser6403_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2018233997510340),0c471d3e-90a5-40dd-b434-100172140241,0504-040911-6ssd4rmw-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 2251)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-04T04:16:14.000Z,141477211871293,azuser6403_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2018233997510340),3e762e57-8388-4bba-8c00-2e764a3af039,0504-040911-6ssd4rmw-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 8, numOutputBytes -> 2814)",null,Databricks-Runtime/18.1.x-photon-scala2.13


2. Query an older version of the table

In [0]:
%sql
SELECT *
FROM patient_delta VERSION AS OF 1;

visit_id,patient_name,city,department,consultation_fee,tests_count,test_cost,total_bill
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,2000,7000
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,7000
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,3500
104,Priya Nair,Bangalore,Cardiology,5000,2,4000,9000
105,Vikram Singh,Chennai,Neurology,7000,1,2000,9000
106,Ananya Das,Kolkata,Orthopedics,3000,3,6000,9000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,2000,7000
108,Meera Iyer,Bangalore,Dermatology,1500,2,4000,5500
109,New Patient,Delhi,Cardiology,4000,1,2000,6000


3. Explain the effect of VACUUM
 -- Removes old unused data files after retention

dry run

In [0]:
%sql
VACUUM patient_delta DRY RUN;

path


PART 5 — PARQUET → DELTA

Save Parquet

In [0]:
df.write.mode("overwrite").parquet("/tmp/patient_parquet")

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-4795362430170917>, line 1
----> 1 df.write.mode("overwrite").parquet("/tmp/patient_parquet")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:779, in DataFrameWriter.parquet(self, path, mode, partitionBy, compression)
    777     self.partitionBy(partitionBy)
    778 self._set_opts(compression=compression)
--> 779 self.format("parquet").save(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site

Convert

In [0]:
%sql
CONVERT TO DELTA parquet.`/tmp/patient_parquet`;

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-4795362430170921>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CONVERT TO DELTA parquet.`/tmp/patient_parquet`;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.driver_activ

Validate

In [0]:
%sql
DESCRIBE DETAIL delta.`/tmp/patient_parquet`;

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-4795362430170924>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'DESCRIBE DETAIL delta.`/tmp/patient_parquet`;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.driver_activity

PART 6 — INCREMENTAL LOAD
Target Table

In [0]:
%sql
CREATE TABLE patient_target
USING DELTA AS SELECT * FROM patient_delta;

num_affected_rows,num_inserted_rows


Daily Updates Dataset

In [0]:
updates = spark.createDataFrame([
(101,"Arjun Reddy","Hyderabad","Cardiology",5500,2),
(120,"New Entry","Delhi","Neurology",7000,1)
], columns[:-2])

Incremental Logic (UPSERT)

In [0]:
%sql
MERGE INTO patient_target t
USING updates s
ON t.visit_id=s.visit_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4795362430170931>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'MERGE INTO patient_target t\nUSING updates s\nON t.visit_id=s.visit_id\nWHEN MATCHED THEN UPDATE SET *\nWHEN NOT MATCHED THEN INSERT *;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic

PART 7 — DELTA LIVE TABLES (DLT)
Bronze

In [0]:
import dlt

@dlt.table
def bronze_visits():
    return spark.table("patient_delta")

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-4795362430170933>, line 1
----> 1 import dlt
      3 @dlt.table
      4 def bronze_visits():
      5     return spark.table("patient_delta")

ModuleNotFoundError: No module named 'dlt'

Silver

In [0]:
@dlt.table
def silver_visits():
    df = dlt.read("bronze_visits")
    return df.filter("consultation_fee > 0")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4795362430170935>, line 1
----> 1 @dlt.table
      2 def silver_visits():
      3     df = dlt.read("bronze_visits")
      4     return df.filter("consultation_fee > 0")

NameError: name 'dlt' is not defined

Gold

In [0]:
@dlt.table
def gold_summary():
    df = dlt.read("silver_visits")
    return df.groupBy("department")\
             .agg(sum("total_bill").alias("revenue"))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4795362430170937>, line 1
----> 1 @dlt.table
      2 def gold_summary():
      3     df = dlt.read("silver_visits")
      4     return df.groupBy("department")\
      5              .agg(sum("total_bill").alias("revenue"))

NameError: name 'dlt' is not defined

PART 8 — UNITY CATALOG
Create Catalog

In [0]:
%sql
CREATE CATALOG healthcare_cat;

Create Schema

In [0]:
%sql
CREATE SCHEMA healthcare_cat.analytics;

Create Table

In [0]:
%sql
CREATE TABLE healthcare_cat.analytics.patient_table
USING DELTA
AS SELECT * FROM patient_delta;

num_affected_rows,num_inserted_rows


Load Data

In [0]:
%sql
INSERT INTO healthcare_cat.analytics.patient_table
SELECT * FROM patient_delta;

num_affected_rows,num_inserted_rows
8,8


PART 9 — DATA GOVERNANCE
Discover Tables

In [0]:

%sql
SHOW TABLES IN healthcare_cat.analytics;

database,tableName,isTemporary
analytics,patient_table,false
,patient_visits,true


Derived Table

In [0]:
%sql
CREATE TABLE healthcare_cat.analytics.high_value_patients AS
SELECT * FROM healthcare_cat.analytics.patient_table
WHERE total_bill > 8000;

num_affected_rows,num_inserted_rows


Access Control

In [0]:
%sql
GRANT SELECT
ON TABLE healthcare_cat.analytics.patient_table
TO `data_analyst`;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4795362430170949>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'GRANT SELECT\nON TABLE healthcare_cat.analytics.patient_table\nTO `data_analyst`;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as

Final Capstone
STEP 1 — Create Unity Catalog Structure

Create Catalog,schema

In [0]:
%sql
CREATE CATALOG healthcare_catalog;
CREATE SCHEMA healthcare_catalog.raw;
CREATE SCHEMA healthcare_catalog.analytics;
USE CATALOG healthcare_catalog;
USE SCHEMA raw;

STEP 2 — Raw Data (BRONZE Layer)

In [0]:
data = [
(101,"Arjun","Hyderabad","Cardiology",5000,1),
(102,"Sneha","Delhi","Orthopedics",3000,2),
(103,"Rahul","Mumbai","Dermatology",1500,1),
(104,"Priya","Bangalore","Cardiology",5000,2)
]

cols=[
"visit_id","patient_name","city",
"department","consultation_fee","tests_count"
]

df = spark.createDataFrame(data,cols)

In [0]:
df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("healthcare_catalog.raw.bronze_visits")

STEP 3 — Clean Data (SILVER Layer)

In [0]:
from pyspark.sql.functions import *

silver_df = df \
.withColumn("department", upper("department")) \
.withColumn("test_cost", col("tests_count")*2000) \
.withColumn("total_bill",
            col("consultation_fee")+col("test_cost"))

In [0]:
silver_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("healthcare_catalog.analytics.silver_visits")

STEP 4 — Analytics (GOLD Layer)

In [0]:
%sql
CREATE TABLE healthcare_catalog.analytics.gold_summary AS
SELECT
city,
department,
COUNT(*) AS total_patients,
SUM(tests_count) AS total_tests,
SUM(total_bill) AS total_revenue
FROM healthcare_catalog.analytics.silver_visits
GROUP BY city, department;

num_affected_rows,num_inserted_rows


STEP 5 — Incremental Load

In [0]:
updates = [
(104,"Priya","Bangalore","Cardiology",6000,2),
(105,"Vikram","Chennai","Neurology",7000,1)
]

updates_df = spark.createDataFrame(updates,cols)

updates_df.createOrReplaceTempView("daily_updates")

In [0]:
%sql
MERGE INTO healthcare_catalog.analytics.silver_visits t
USING daily_updates s
ON t.visit_id = s.visit_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4795362430170963>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'MERGE INTO healthcare_catalog.analytics.silver_visits t\nUSING daily_updates s\nON t.visit_id = s.visit_id\n\nWHEN MATCHED THEN\nUPDATE SET *\n\nWHEN NOT MATCHED THEN\nINSERT *\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntim

STEP 6 — Build DLT Pipeline 

In [0]:
%sql
CREATE OR REFRESH LIVE TABLE bronze_visits
AS
SELECT * FROM healthcare_catalog.raw.bronze_visits;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4795362430170964>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REFRESH LIVE TABLE bronze_visits\nAS\nSELECT * FROM healthcare_catalog.raw.bronze_visits;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 excep

In [0]:
%sql
CREATE OR REFRESH LIVE TABLE silver_visits
AS
SELECT *,
consultation_fee + tests_count*2000 AS total_bill
FROM LIVE.bronze_visits;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4795362430170966>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REFRESH LIVE TABLE silver_visits\nAS\nSELECT *,\nconsultation_fee + tests_count*2000 AS total_bill\nFROM LIVE.bronze_visits;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.

In [0]:
%sql
CREATE OR REFRESH LIVE TABLE gold_summary
AS
SELECT
city,
department,
COUNT(*) total_patients,
SUM(total_bill) total_revenue
FROM LIVE.silver_visits
GROUP BY city, department;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4795362430170967>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REFRESH LIVE TABLE gold_summary\nAS\nSELECT\ncity,\ndepartment,\nCOUNT(*) total_patients,\nSUM(total_bill) total_revenue\nFROM LIVE.silver_visits\nGROUP BY city, department;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/d

Apply Access Control

In [0]:
%sql
GRANT SELECT
ON TABLE healthcare_catalog.analytics.gold_summary
TO `analyst`;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4795362430170968>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'GRANT SELECT\nON TABLE healthcare_catalog.analytics.gold_summary\nTO `analyst`;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e

View Audit Logs

In [0]:
%sql
SELECT *
FROM system.access.audit
ORDER BY event_time DESC;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4795362430170970>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT *\nFROM system.access.audit\nORDER BY event_time DESC;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self